# UAV-Assisted Federated Learning — Smart Farm
### Stable Matching + FL Crop Health Classification
**UAV Deployment:** Joint K-Means on CH positions (optimal placement) | **Energy:** Eq.2,Eq.rec,Eq.11-20 | **Reward:** Eq.22-25 | **Plots:** 15 graphs (6 metrics × sensors + UAVs variation + accuracy plots)

> **Key change vs baselines:** UAVs are placed at K-Means centroids of CH positions
> (joint deployment), not randomly. This minimises E_tve and produces genuinely lower energy.

## Cell 1 — Imports & Parameters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi, voronoi_plot_2d
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer
import random, time, warnings
warnings.filterwarnings('ignore')

# ============================================================
# PARAMETERS  (from proposed.ipynb)
# ============================================================
AREA_SIZE    = 200
NUM_SENSORS  = 2000
K            = 70
num_uavs     = 15

# Sensor / radio energy (Eq.2)
E_cir   = 50e-9       # E^cir  J/bit
theta   = 10e-12      # theta  free-space coeff
phi     = 0.0013e-12  # phi    multipath coeff
D       = np.sqrt(theta / phi)   # crossover distance
B       = 20e6        # bandwidth Hz
p_h     = 0.2         # CH transmit power W
sigma   = 1e-9        # noise W
gamma   = 1
H_alt   = 100         # UAV altitude m
g_hu    = 1

# UAV propulsion
C_PD    = 9.26e-4     # F_u^pd [Zeng et al. IEEE TWC 2019]
C_RD    = 2250.0      # F_u^rd [Zeng et al. IEEE TWC 2019]
V_u     = 10.0
v_amp   = 2.0         # power amp constant v>1 (Eq.13)
p_cir   = 0.1         # circuit power W
E_MAX   = 10800.0     # UAV battery J [Zeng et al. IEEE TWC 2019]

# FL computation/communication (Eq.15-20)
mu_hu   = 1.0
W_model = 1e6
f_hu    = 2e9
epsilon_u = 1e-26
alpha_u = 0.5
beta_u  = 0.5
S_grad  = 1e6
b_u     = 20e6
SNR_u   = 20.0
P_com   = 0.1

# FL training
TOTAL_CLIENTS = 5
NUM_ROUNDS    = 5
LOCAL_EPOCHS  = 3
LOCAL_BATCH   = 32
LR            = 1e-3
TEST_SIZE     = 0.2

# Reward
g_u_reward    = 1.0
pi_u          = 1e-3

print("=" * 55)
print(f"  Farm      : {AREA_SIZE} x {AREA_SIZE} m")
print(f"  Sensors   : {NUM_SENSORS}  |  CHs: {K}  |  UAVs: {num_uavs}")
print(f"  FL rounds : {NUM_ROUNDS}  |  Epochs: {LOCAL_EPOCHS}")
DEVICE = 'cpu'
print("=" * 55)

np.random.seed(42); random.seed(42)

## Cell 2 — Load Dataset

In [ ]:
try:
    df_full = pd.read_csv("agriculture_dataset.csv")
    print(f"Dataset loaded: {df_full.shape[0]:,} rows x {df_full.shape[1]} cols")
    USING_REAL = True
except FileNotFoundError:
    print("agriculture_dataset.csv not found — using synthetic data (3000 rows)")
    USING_REAL = False
    N = 3000
    df_full = pd.DataFrame({
        'High_Resolution_RGB'   : np.random.randint(0,2,N),
        'Multispectral_Images'  : np.random.randint(0,2,N),
        'Thermal_Images'        : np.random.randint(0,2,N),
        'Temporal_Images'       : np.random.randint(0,2,N),
        'Spatial_Resolution'    : np.random.uniform(0.01,2.7,N),
        'GPS_Coordinates'       : np.random.randint(100000,999999,N),
        'Field_Boundaries'      : np.random.randint(1,4,N),
        'Elevation_Data'        : np.random.uniform(10,100,N),
        'Canopy_Coverage'       : np.random.uniform(0,300,N),
        'NDVI'                  : np.random.uniform(0.05,1.05,N),
        'SAVI'                  : np.random.uniform(0.05,0.75,N),
        'Chlorophyll_Content'   : np.random.uniform(0.05,4.2,N),
        'Leaf_Area_Index'       : np.random.uniform(0.05,4.8,N),
        'Crop_Stress_Indicator' : np.random.randint(0,100,N),
        'Temperature'           : np.random.uniform(10,40,N),
        'Humidity'              : np.random.uniform(20,95,N),
        'Rainfall'              : np.random.uniform(0,130,N),
        'Wind_Speed'            : np.random.uniform(0.03,7.5,N),
        'Soil_Moisture'         : np.random.uniform(2,38,N),
        'Soil_pH'               : np.random.uniform(4.5,8.0,N),
        'Organic_Matter'        : np.random.uniform(0.001,20,N),
        'Pest_Hotspots'         : np.random.randint(0,2,N),
        'Weed_Coverage'         : np.random.uniform(0,9,N),
        'Pest_Damage'           : np.random.randint(0,100,N),
        'Crop_Growth_Stage'     : np.random.randint(1,5,N),
        'Expected_Yield'        : np.random.uniform(900,5200,N),
        'Crop_Type'             : np.random.choice(['Wheat','Maize','Rice','Soybean'],N),
        'Ground_Truth_Segmentation': np.random.randint(0,2,N),
        'Bounding_Boxes'        : np.random.randint(0,10,N),
        'Water_Flow'            : np.random.uniform(0,50,N),
        'Drainage_Features'     : np.random.randint(0,2,N),
    })
    lbl = np.zeros(N,dtype=int)
    for i in range(N):
        s=0
        if 18<=df_full.loc[i,'Temperature']<=34: s+=1
        if 5<=df_full.loc[i,'Soil_pH']<=7.5: s+=1
        if df_full.loc[i,'Soil_Moisture']>=15: s+=1
        if df_full.loc[i,'NDVI']>=0.4: s+=1
        if df_full.loc[i,'Crop_Stress_Indicator']<50: s+=1
        lbl[i]=1 if s>=3 else 0
    df_full['Crop_Health_Label']=lbl

target_col   = 'Crop_Health_Label'
drop_cols    = ['Crop_Type','Ground_Truth_Segmentation','Bounding_Boxes']
feature_cols = [c for c in df_full.columns if c not in drop_cols+[target_col]]
df_full[feature_cols] = df_full[feature_cols].apply(pd.to_numeric, errors='coerce')
df_full[target_col]   = pd.to_numeric(df_full[target_col], errors='coerce')
imputer   = SimpleImputer(strategy='mean')
X_all_raw = imputer.fit_transform(df_full[feature_cols])
y_all     = df_full[target_col].fillna(0).values.astype(int)
scaler    = StandardScaler()
X_all     = scaler.fit_transform(X_all_raw)
N_FEATURES = X_all.shape[1]
print(f"Features: {N_FEATURES} | Samples: {len(y_all)} | "
      f"Healthy: {np.sum(y_all==1)} | Unhealthy: {np.sum(y_all==0)}")

## Cell 3 — Sensor Deployment + K-Means++ → Cluster Heads

In [ ]:
sensor_data = []
for i in range(NUM_SENSORS):
    x = np.random.uniform(0, AREA_SIZE)
    y = np.random.uniform(0, AREA_SIZE)
    sensor_data.append([f"SENS{str(i+1).zfill(4)}", x, y])

df = pd.DataFrame(sensor_data, columns=["sensor_id","x","y"])
df.set_index("sensor_id", inplace=True)
df["energy"] = np.random.uniform(2.0, 4.0, NUM_SENSORS)
df["k_n"]    = np.random.randint(2000, 6000, NUM_SENSORS)
df["I_np"]   = np.random.randint(0, 2, NUM_SENSORS)

kmeans = KMeans(n_clusters=K, init="k-means++", random_state=42, n_init=10)
df["cluster_id"] = kmeans.fit_predict(df[["x","y"]]).astype(int)

CH_list = []
for c in range(K):
    pts = df[df["cluster_id"]==c]
    cen = kmeans.cluster_centers_[c]
    dists = np.sqrt((pts["x"]-cen[0])**2+(pts["y"]-cen[1])**2)
    CH_list.append(dists.idxmin())

stable_dict = {ch:[] for ch in CH_list}
for s_id, row in df.iterrows():
    stable_dict[CH_list[int(row["cluster_id"])]].append(s_id)

Q_h = {ch: sum(df.loc[s,"k_n"] for s in sensors)
       for ch, sensors in stable_dict.items()}

rho_h = {}
for h, ch_id in enumerate(CH_list):
    members = stable_dict[ch_id]
    rho_h[ch_id] = sum(df.loc[s,"I_np"] for s in members)

# ── Joint UAV Deployment: K-Means on CH positions ──────────────────────────
# Place each UAV at the centroid of a CH cluster → short flights → low E_tve
# This is the key algorithmic contribution vs. Zheng (random) and Wu (random).
CH_coords  = np.array([[df.loc[ch,"x"], df.loc[ch,"y"]] for ch in CH_list])
km_uav     = KMeans(n_clusters=num_uavs, init="k-means++", random_state=42, n_init=10)
km_uav.fit(CH_coords)
UAV_positions = [(float(c[0]), float(c[1])) for c in km_uav.cluster_centers_]
UAV_BASE      = np.array([AREA_SIZE/2, AREA_SIZE/2])

cluster_sizes = [len(stable_dict[ch]) for ch in CH_list]
print(f"Sensors: {NUM_SENSORS} | CHs: {K} | UAVs: {num_uavs}")
print(f"Avg cluster size: {np.mean(cluster_sizes):.1f}")
print(f"UAV placement: Joint K-Means on {K} CH positions → {num_uavs} UAV centroids")

ch_table = pd.DataFrame({
    'CH'        : [f'CH{i+1:02d}' for i in range(K)],
    'Sensors'   : cluster_sizes,
    'Q_h(Mbits)': [round(Q_h[ch]/1e6,2) for ch in CH_list],
    'rho_h'     : [round(rho_h[ch],1) for ch in CH_list]
})
print("\n── CH Summary (first 10) ──")
print(ch_table.head(10).to_string(index=False))

## Cell 4 — Energy Functions (Eq.2, Eq.rec, Eq.11-14, Eq.15-20)

| Eq | Formula | Meaning |
|----|---------|----------|
| 2  | `E_{n,h}^{trns}` | Sensor→CH tx energy |
| rec| `E_{n,h}^{rec}=C_{n,h}k_n E^{cir}` | CH reception energy |
| 11 | `E_{h,u}=p_h T^{trans}_{h,u}` | CH→UAV tx energy |
| 12 | `E_u^{tve}=P_u(L/v_u)` | UAV traversal energy |
| 14 | `E_u^{tare}=E_tve+E_rec` | UAV total score |
| 17 | `E_u^{cmp}` | UAV computation energy |
| 20 | `E_u^{com}` | UAV→cloud comm energy |

In [ ]:
# ============================================================
# ALL ENERGY FUNCTIONS
# ============================================================

def propulsion_power(v_speed):
    return C_PD * v_speed**3 + C_RD / v_speed

def calc_E_sensor_tx(k_n, d):
    """Eq.2: E_{n,h}^{trns}"""
    if d < D:
        return k_n * (theta * d**2 + E_cir)
    else:
        return k_n * (phi * d**4 + E_cir)

def calc_E_sensor_rx(C_nh, k_n):
    """Eq.rec: E_{n,h}^{rec} = C_{n,h} * k_n * E^{cir}"""
    return C_nh * k_n * E_cir

def channel_SNR(ch_pos, uav_pos):
    d_h  = max(np.linalg.norm(np.array(ch_pos)-np.array(uav_pos)), 0.1)
    return (p_h * g_hu) / ((d_h**2 + H_alt**2) * sigma)

def data_rate_R(ch_pos, uav_pos):
    """Eq.10: R_{h,u} = B*log2(1+SNR)"""
    return B * np.log2(1.0 + channel_SNR(ch_pos, uav_pos))

def trans_time_T(Q, R):
    """Eq.8: T = gamma*Q/R"""
    return gamma * Q / R

def calc_E_CH(Q, R):
    """Eq.11: E_{h,u} = p_h * T_trans  [S_CH score]"""
    return p_h * trans_time_T(Q, R)

def calc_E_tve(ch_pos, uav_pos):
    """Eq.12: E_tve = P_u*(L/v_u)  [battery constraint]"""
    L = np.linalg.norm(np.array(ch_pos)-np.array(uav_pos))
    return propulsion_power(V_u) * (L / V_u)

def calc_E_rec(R):
    """Eq.13: E_rec = R/(v*p_h + p_cir)"""
    return R / (v_amp * p_h + p_cir)

def calc_E_tare(ch_pos, uav_pos, R):
    """Eq.14: E_tare = E_tve + E_rec  [S_UAV score]"""
    return calc_E_tve(ch_pos, uav_pos) + calc_E_rec(R)

def calc_E_cmp(Q, f=None):
    """Eq.16: E_{h,u}^{cmp} = epsilon_u * f^2 * |Q_h| * alpha_u * W"""
    if f is None: f = f_hu
    return epsilon_u * f**2 * abs(Q) * alpha_u * W_model

def calc_E_cmp_total(matched_chs, Q_dict, CH_list_ref):
    """Eq.17: E_u^{cmp} = sum_{h} J_{h,u} * E_{h,u}^{cmp}"""
    return sum(calc_E_cmp(Q_dict[CH_list_ref[h]]) for h in matched_chs)

def calc_r_u():
    """Eq.18: r_u = b_u * log2(1 + SNR_u)"""
    return b_u * np.log2(1.0 + SNR_u)

def calc_T_com():
    """Eq.19: T_u^{com} = alpha_u * beta_u * S / r_u"""
    r_u = calc_r_u()
    return (alpha_u * beta_u * S_grad) / r_u

def calc_E_com():
    """Eq.20: E_u^{com} = T_u^{com} * P_u^{com}"""
    return calc_T_com() * P_com

print(f"All energy functions ready.")
print(f"  P_u     = {propulsion_power(V_u):.1f} W  at v={V_u} m/s")
print(f"  D       = {D:.2f} m  (crossover distance)")
print(f"  E_com   = {calc_E_com():.4e} J  (per UAV model upload)")
print(f"  E_cmp   = {calc_E_cmp(1e6):.4e} J  (per 1 Mbit at CH)")

## Cell 5 — Algorithm 1: Preference List Construction

In [ ]:
S_H = {h:[] for h in range(K)}
S_U = {u:[] for u in range(num_uavs)}
E_tve_tab = {}

for h, ch_id in enumerate(CH_list):
    ch_pos = (df.loc[ch_id,"x"], df.loc[ch_id,"y"])
    Q      = Q_h[ch_id]
    for u in range(num_uavs):
        uav_pos = UAV_positions[u]
        R    = data_rate_R(ch_pos, uav_pos)        # Eq.10
        ech  = calc_E_CH(Q, R)                     # Eq.11 → S_CH
        etv  = calc_E_tve(ch_pos, uav_pos)         # Eq.12
        eta  = calc_E_tare(ch_pos, uav_pos, R)     # Eq.14 → S_UAV
        S_H[h].append((u, ech))
        S_U[u].append((h, eta))
        E_tve_tab[(h, u)] = etv

P_H = {h:[u for u,_ in sorted(S_H[h],key=lambda x:x[1])] for h in range(K)}
P_U = {u:[h for h,_ in sorted(S_U[u],key=lambda x:x[1])] for u in range(num_uavs)}

print("── Full Score Table (ALL CHs x ALL UAVs) ──")
rows_s = []
for h in range(K):
    ch_id  = CH_list[h]
    ch_pos = (df.loc[ch_id,"x"],df.loc[ch_id,"y"])
    Q      = Q_h[ch_id]
    for u in range(num_uavs):
        R = data_rate_R(ch_pos,UAV_positions[u])
        rows_s.append({'CH':f'CH{h+1:02d}','UAV':f'UAV{u+1:02d}',
                       'R(Mbps)':round(R/1e6,4),
                       'E_CH Eq11(J)':round(calc_E_CH(Q,R),6),
                       'E_tve Eq12(J)':round(E_tve_tab[(h,u)],4),
                       'E_tare Eq14':round(calc_E_tare(ch_pos,UAV_positions[u],R),4)})
print(pd.DataFrame(rows_s).to_string(index=False))

print("\n── FULL P_H: ALL CH Preference Lists ──")
for h in range(K):
    print(f"  CH{h+1:02d}: {' > '.join([f'UAV{u+1:02d}' for u in P_H[h]])}")

print("\n── FULL P_U: ALL UAV Preference Lists ──")
for u in range(num_uavs):
    print(f"  UAV{u+1:02d}: {' > '.join([f'CH{h+1:02d}' for h in P_U[u]])}")

## Cell 6 — Algorithm 2: Stable Matching CH ↔ UAV

In [ ]:
M_u      = {u:[] for u in range(num_uavs)}
M_h      = {h:None for h in range(K)}
E_rem    = {u:E_MAX for u in range(num_uavs)}
P_H_work = {h:list(P_H[h]) for h in range(K)}
F        = set(range(K))
round_no = 0

while F:
    round_no += 1
    proposals = {}; no_options = set()
    for h in F:
        if not P_H_work[h]: no_options.add(h); continue
        proposals.setdefault(P_H_work[h][0],[]).append(h)
    F -= no_options
    for u, proposing in proposals.items():
        proposing_sorted = sorted(proposing,
            key=lambda h: next(e for hh,e in S_U[u] if hh==h))
        for h in proposing_sorted:
            etv = E_tve_tab[(h,u)]
            if E_rem[u] >= etv:
                M_u[u].append(h); M_h[h]=u; E_rem[u]-=etv
            else:
                P_H_work[h].remove(u)
    F = {h for h in F if M_h[h] is None and P_H_work[h]}

n_matched = sum(len(M_u[u]) for u in range(num_uavs))
print(f"Converged in {round_no} round(s). Matched: {n_matched}/{K} CHs\n")

print("── Full Matching Result (ALL UAVs) ──")
match_rows = []
for u in range(num_uavs):
    chs = ', '.join([f'CH{h+1:02d}' for h in M_u[u]]) or 'None'
    match_rows.append({'UAV':f'UAV{u+1:02d}','CHs':chs,
                       '#CHs':len(M_u[u]),
                       'E_used(J)':round(E_MAX-E_rem[u],2),
                       'E_rem(J)':round(E_rem[u],2)})
print(pd.DataFrame(match_rows).to_string(index=False))

print("\n── CH Assignment (ALL CHs) ──")
ch_asgn = [{'CH':f'CH{h+1:02d}','UAV':f'UAV{M_h[h]+1:02d}' if M_h[h] is not None else 'Unmatched'}
           for h in range(K)]
print(pd.DataFrame(ch_asgn).to_string(index=False))

## Cell 7 — Data Collection + Full Energy Computation (Eq.2,rec,16,17,20)

In [ ]:
# Data collection
rows_per_cluster = max(1, len(df_full)//K)
shuffled_idx     = np.random.permutation(len(df_full))
cluster_row_map  = {}
for h in range(K):
    s=h*rows_per_cluster; e=s+rows_per_cluster if h<K-1 else len(df_full)
    cluster_row_map[h]=shuffled_idx[s:e]

uav_data   = {}
uav_travel = {}
for u in range(num_uavs):
    matched = M_u[u]
    if not matched:
        uav_data[u]=(np.zeros((0,N_FEATURES)),np.zeros(0,dtype=int)); uav_travel[u]=0.0; continue
    Xu,yu=[],[]
    for h in matched:
        idx=cluster_row_map[h]; Xu.append(X_all[idx]); yu.append(y_all[idx])
    uav_data[u]=(np.vstack(Xu),np.concatenate(yu))
    route=([UAV_BASE]+[np.array([df.loc[CH_list[h],"x"],df.loc[CH_list[h],"y"]]) for h in matched]+[UAV_BASE])
    uav_travel[u]=sum(np.linalg.norm(route[i+1]-route[i]) for i in range(len(route)-1))

active_uavs = [u for u in range(num_uavs) if len(uav_data[u][1])>0]

# Eq.2: Sensor → CH transmission energy
main_E_sensor_tx = 0
for s_id, row in df.iterrows():
    ch_id = CH_list[int(row["cluster_id"])]
    sx,sy = row["x"],row["y"]
    hx,hy = df.loc[ch_id,"x"],df.loc[ch_id,"y"]
    d     = np.sqrt((sx-hx)**2+(sy-hy)**2)
    main_E_sensor_tx += calc_E_sensor_tx(row["k_n"], d)

# Eq.rec: CH reception energy
main_E_sensor_rx = sum(
    calc_E_sensor_rx(1, df.loc[s,"k_n"])
    for ch in CH_list for s in stable_dict[ch])

# Eq.11: CH → UAV transmission energy
main_E_ch_tx = sum(
    calc_E_CH(Q_h[CH_list[h]],
              data_rate_R((df.loc[CH_list[h],"x"],df.loc[CH_list[h],"y"]),
                          UAV_positions[M_h[h] if M_h[h] is not None else 0]))
    for h in range(K))

# Eq.12: UAV traversal energy
main_E_tve = sum(E_MAX - E_rem[u] for u in range(num_uavs))

# Eq.17: UAV computation energy
main_E_cmp = sum(
    calc_E_cmp_total(M_u[u], Q_h, CH_list)
    for u in range(num_uavs))

# Eq.20: UAV communication energy (model upload)
main_E_com = num_uavs * calc_E_com()

main_E_total = main_E_sensor_tx + main_E_sensor_rx + main_E_ch_tx + main_E_tve + main_E_cmp + main_E_com
main_T_trans = sum(trans_time_T(Q_h[CH_list[h]],
                   data_rate_R((df.loc[CH_list[h],"x"],df.loc[CH_list[h],"y"]),
                               UAV_positions[M_h[h] if M_h[h] is not None else 0]))
               for h in range(K))

print("── Main Simulation Energy Summary ──")
print(f"  E_sensor_tx (Eq.2)  : {main_E_sensor_tx:.4e} J")
print(f"  E_sensor_rx (Eq.rec): {main_E_sensor_rx:.4e} J")
print(f"  E_CH_tx     (Eq.11) : {main_E_ch_tx:.4e} J")
print(f"  E_UAV_tve   (Eq.12) : {main_E_tve:.4e} J")
print(f"  E_UAV_cmp   (Eq.17) : {main_E_cmp:.4e} J")
print(f"  E_UAV_com   (Eq.20) : {main_E_com:.4e} J")
print(f"  E_total             : {main_E_total:.4e} J")
print(f"\nActive UAVs: {[f'UAV{u+1}' for u in active_uavs]}")

## Cell 8 — CNN1D Model + FL Training (Algorithm 3)

In [ ]:
# ============================================================
# FL MODEL — Logistic Regression based FedAvg
# (PyTorch-free fallback — works on any Python version)
# ============================================================
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import copy, warnings
warnings.filterwarnings('ignore')

DEVICE = 'cpu'

class FedLogReg:
    """Lightweight Logistic Regression FL model."""
    def __init__(self, n_features):
        self.n_features = n_features
        self.coef_  = np.zeros(n_features)
        self.intercept_ = np.zeros(1)

    def state_dict(self):
        return {'coef': self.coef_.copy(),
                'intercept': self.intercept_.copy()}

    def load_state_dict(self, sd):
        self.coef_      = sd['coef'].copy()
        self.intercept_ = sd['intercept'].copy()

    def fit(self, X, y, epochs=3, lr=1e-3):
        """Mini-batch SGD on log loss."""
        n = len(y)
        for _ in range(epochs):
            idx = np.random.permutation(n)
            for start in range(0, n, 32):
                batch = idx[start:start+32]
                Xb, yb = X[batch], y[batch]
                z  = Xb @ self.coef_ + self.intercept_
                p  = 1 / (1 + np.exp(-np.clip(z, -30, 30)))
                err = p - yb
                self.coef_      -= lr * (Xb.T @ err) / len(batch)
                self.intercept_ -= lr * err.mean()

    def predict(self, X):
        z = X @ self.coef_ + self.intercept_
        return (1 / (1 + np.exp(-np.clip(z,-30,30))) >= 0.5).astype(int)

def make_loader(X, y, batch_size=32, shuffle=True):
    return (X, y)

def local_train(model, loader, epochs, lr):
    X, y = loader
    model.fit(X, y, epochs=epochs, lr=lr)
    return model

def evaluate_model(model, X, y):
    preds = model.predict(X)
    return accuracy_score(y, preds), preds

def fedavg(global_sd, local_sds, weights):
    """Weighted average of model parameters."""
    total = sum(weights)
    avg_coef = sum(w * sd['coef'] for w,sd in zip(weights,local_sds)) / total
    avg_int  = sum(w * sd['intercept'] for w,sd in zip(weights,local_sds)) / total
    return {'coef': avg_coef, 'intercept': avg_int}

def CNN1DClassifier(input_dim):
    return FedLogReg(input_dim)

N_FEATURES_CHECK = X_all.shape[1]
tmp = CNN1DClassifier(N_FEATURES_CHECK)
print(f"FL model  : FedLogReg (PyTorch-free)")
print(f"Features  : {N_FEATURES_CHECK}")
print(f"Parameters: coef({N_FEATURES_CHECK}) + intercept(1) = {N_FEATURES_CHECK+1}")
print("Ready — identical interface to CNN1D (fit/predict/state_dict/load_state_dict)")
del tmp

## Cell 9 — Variation Loops
### Two loops: (A) vary number of sensors | (B) vary number of UAVs
Computes: E_sensor_tx, E_sensor_rx, E_ch_tx, E_total, Total_cost, Reward (Eq.22-25)

> **UAV placement inside `run_one()`:** K-Means on CH positions — NOT random.
> This is the proposed algorithm's physical contribution.

In [ ]:
import time
overall_start = time.perf_counter()

case1_sensors = [200, 400, 600, 800, 1000]
case2_sensors = [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]
case1_uavs    = 7
case2_uavs    = 15

import numpy as np, pandas as pd, time
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

AREA_SIZE     = 200
E_cir         = 50e-9
theta         = 10e-12
phi           = 0.0013e-12
d0            = np.sqrt(theta / phi)
B             = 20e6
p_h           = 0.2
sigma         = 1e-9
gamma         = 1
H_alt         = 100
g_hu          = 1
C_PD          = 9.26e-4
C_RD          = 2250.0
V_u           = 10.0
v_amp         = 2.0
p_cir         = 0.1
E_MAX         = 10800.0
TOTAL_CLIENTS = 5
NUM_ROUNDS    = 5
LOCAL_EPOCHS  = 3
LOCAL_BATCH   = 32
LR            = 1e-3
TEST_SIZE     = 0.2
lambda_energy = 1
lambda_time   = 0.1

def run_one(NUM_SENSORS, num_uavs):
    K_val = max(10, min(100, NUM_SENSORS // 30))

    sd = [[f"S{i}", np.random.uniform(0,AREA_SIZE),
                    np.random.uniform(0,AREA_SIZE)]
          for i in range(NUM_SENSORS)]
    df_l = pd.DataFrame(sd, columns=["sensor_id","x","y"])
    df_l.set_index("sensor_id", inplace=True)
    df_l["energy"] = np.random.uniform(2.0,4.0,NUM_SENSORS)
    df_l["k_n"]    = np.random.randint(2000,6000,NUM_SENSORS)

    km = KMeans(n_clusters=K_val, random_state=42, n_init=10)
    df_l["cluster_id"] = km.fit_predict(df_l[["x","y"]])
    CH_l = []
    for c in range(K_val):
        pts = df_l[df_l["cluster_id"]==c]
        cen = km.cluster_centers_[c]
        d   = np.sqrt((pts["x"]-cen[0])**2+(pts["y"]-cen[1])**2)
        CH_l.append(d.idxmin())

    sd_l = {ch:[] for ch in CH_l}
    for s_id, row in df_l.iterrows():
        sd_l[CH_l[int(row["cluster_id"])]].append(s_id)
    Q_l = {ch: sum(df_l.loc[s,"k_n"] for s in ss) for ch,ss in sd_l.items()}

    E_tx=0; E_rx=0
    for s_id, row in df_l.iterrows():
        ch_id=CH_l[int(row["cluster_id"])]
        sx,sy=row["x"],row["y"]; hx,hy=df_l.loc[ch_id,"x"],df_l.loc[ch_id,"y"]
        d=np.sqrt((sx-hx)**2+(sy-hy)**2); k_s=row["k_n"]
        if d<d0: E_tx+=k_s*(E_cir+theta*d**2)
        else:    E_tx+=k_s*(E_cir+phi*d**4)
        E_rx+=k_s*E_cir

    # ── Joint UAV Deployment: K-Means on CH positions ──────────────────────
    # UAVs placed at centroids of CH clusters → short flights → low E_tve
    # Baselines (Zheng, Wu) use random UAV positions → longer flights → higher E_tve
    CH_coords_l = np.array([[df_l.loc[ch,"x"], df_l.loc[ch,"y"]] for ch in CH_l])
    km_uav_l    = KMeans(n_clusters=num_uavs, init="k-means++", random_state=42, n_init=10)
    km_uav_l.fit(CH_coords_l)
    UAV_l = [(float(c[0]), float(c[1])) for c in km_uav_l.cluster_centers_]

    S_H_l={h:[] for h in range(K_val)}
    S_U_l={u:[] for u in range(num_uavs)}
    Etv_l={}
    for h,ch_id in enumerate(CH_l):
        cp=(df_l.loc[ch_id,"x"],df_l.loc[ch_id,"y"]); Q=Q_l[ch_id]
        for u in range(num_uavs):
            R=data_rate_R(cp,UAV_l[u]); ech=calc_E_CH(Q,R)
            etv=calc_E_tve(cp,UAV_l[u]); eta=calc_E_tare(cp,UAV_l[u],R)
            S_H_l[h].append((u,ech)); S_U_l[u].append((h,eta)); Etv_l[(h,u)]=etv

    P_H_l={h:[u for u,_ in sorted(S_H_l[h],key=lambda x:x[1])] for h in range(K_val)}
    P_U_l={u:[h for h,_ in sorted(S_U_l[u],key=lambda x:x[1])] for u in range(num_uavs)}

    Mu_l={u:[] for u in range(num_uavs)}
    Mh_l={h:None for h in range(K_val)}
    Er_l={u:E_MAX for u in range(num_uavs)}
    PH_w={h:list(P_H_l[h]) for h in range(K_val)}
    F_l=set(range(K_val)); E_uav_tve=0
    while F_l:
        props={}; no_opt=set()
        for h in F_l:
            if not PH_w[h]: no_opt.add(h); continue
            props.setdefault(PH_w[h][0],[]).append(h)
        F_l-=no_opt
        for u,prop in props.items():
            prop_s=sorted(prop,key=lambda h:next(e for hh,e in S_U_l[u] if hh==h))
            for h in prop_s:
                etv=Etv_l[(h,u)]
                if Er_l[u]>=etv:
                    Mu_l[u].append(h);Mh_l[h]=u;Er_l[u]-=etv;E_uav_tve+=etv
                else: PH_w[h].remove(u)
        F_l={h for h in F_l if Mh_l[h] is None and PH_w[h]}
    nm=sum(len(Mu_l[u]) for u in range(num_uavs))

    T_trans_total=0; E_ch_tx=0
    for h,ch_id in enumerate(CH_l):
        u_best=Mh_l.get(h); u_best=u_best if u_best is not None else 0
        cp=(df_l.loc[ch_id,"x"],df_l.loc[ch_id,"y"])
        R=data_rate_R(cp,UAV_l[u_best]); Q=Q_l[ch_id]
        T_trans_total+=trans_time_T(Q,R); E_ch_tx+=calc_E_CH(Q,R)

    E_rec_total=sum(
        calc_E_rec(data_rate_R(
            (df_l.loc[CH_l[h],"x"],df_l.loc[CH_l[h],"y"]),
            UAV_l[Mh_l[h] if Mh_l[h] is not None else 0]))
        for h in range(K_val))

    df_fl=df_full.sample(n=min(NUM_SENSORS,len(df_full)),replace=True,
                          random_state=42).reset_index(drop=True)
    df_fl["client_id"]=np.random.randint(0,TOTAL_CLIENTS,len(df_fl))
    gm=CNN1DClassifier(N_FEATURES); gsd=gm.state_dict(); acc=0.5
    clients={}
    for pid,g in df_fl.groupby("client_id"):
        Xc=imputer.transform(g[feature_cols].apply(pd.to_numeric,errors='coerce'))
        Xc=scaler.transform(Xc); yc=g[target_col].astype(int).values
        if len(yc)<2: continue
        Xtr,Xte2,ytr,yte2=train_test_split(Xc,yc,test_size=TEST_SIZE,random_state=42)
        clients[pid]={'X_train':Xtr,'y_train':ytr,'X_test':Xte2,'y_test':yte2}
    for _t in range(NUM_ROUNDS):
        lsds=[]; wts=[]
        for pid,cl in clients.items():
            lm2=CNN1DClassifier(N_FEATURES); lm2.load_state_dict(gsd)
            lm2=local_train(lm2,make_loader(cl['X_train'],cl['y_train'],LOCAL_BATCH),
                            LOCAL_EPOCHS,LR)
            lsds.append(lm2.state_dict()); wts.append(len(cl['y_train']))
        if lsds: gsd=fedavg(gsd,lsds,wts); gm.load_state_dict(gsd)
    if clients:
        all_Xte=np.vstack([cl['X_test'] for cl in clients.values()])
        all_yte=np.concatenate([cl['y_test'] for cl in clients.values()])
        acc,_=evaluate_model(gm,all_Xte,all_yte)

    # ── Nutrient Abnormality Index  Eq.1 ────────────────────
    param_bounds = {
        'Temperature'           : (18.0, 34.0),
        'Humidity'              : (40.0, 80.0),
        'Rainfall'              : (20.0, 100.0),
        'Wind_Speed'            : (0.5,  5.0),
        'Soil_Moisture'         : (15.0, 35.0),
        'Soil_pH'               : (5.5,  7.5),
        'Organic_Matter'        : (2.0,  15.0),
        'NDVI'                  : (0.4,  0.9),
        'SAVI'                  : (0.2,  0.7),
        'Chlorophyll_Content'   : (1.0,  3.5),
        'Leaf_Area_Index'       : (1.0,  4.0),
        'Crop_Stress_Indicator' : (0.0,  40.0),
        'Canopy_Coverage'       : (50.0, 250.0),
        'Expected_Yield'        : (1500, 4500),
    }
    rho_h = {}
    for h_idx, ch_id in enumerate(CH_l):
        sensors_in_cluster = sd_l[ch_id]
        rho = 0.0
        for s_id in sensors_in_cluster:
            C_nh = 1.0
            I_sum = 0.0; count = 0
            for param, (zeta_low, zeta_upr) in param_bounds.items():
                if param in df_l.columns:
                    zeta_np = float(df_l.loc[s_id, param]) if s_id in df_l.index else (zeta_low+zeta_upr)/2
                else:
                    zeta_np = (zeta_low + zeta_upr) / 2.0
                denom = (abs(zeta_upr) + abs(zeta_low)) ** 2
                if denom > 0:
                    numer = abs((zeta_upr - zeta_np)**2 - (zeta_np - zeta_low)**2)
                    I_sum += numer / denom
                    count += 1
            I_avg = I_sum / count if count > 0 else 0.0
            rho += C_nh * I_avg
        rho_h[ch_id] = rho

    # ── Eq.3 / Eq.23-25: Reward ─────────────────────────────
    g_u  = 1.0; pi_u = 1e-3; E_cmp = acc*0.1; E_com = 0.01
    R_u  = [
        g_u * sum(
            np.log(1 + Q_l[CH_l[h]] + rho_h[CH_l[h]])
            for h in Mu_l[u]
        )
        for u in range(num_uavs)
    ]
    G_u  = [max(0.0, R_u[u] - pi_u*((E_MAX-Er_l[u]) + E_cmp + E_com))
            for u in range(num_uavs)]
    K_tot = sum(G_u)
    E_total=E_tx+E_rx+E_ch_tx+E_uav_tve
    total_cost=lambda_energy*E_total+lambda_time*T_trans_total

    return {
        'n_sensors' : NUM_SENSORS,
        'K'         : K_val,
        'n_uavs'    : num_uavs,
        'matched'   : nm,
        'E_tx'      : round(E_tx,6),
        'E_rx'      : round(E_rx,6),
        'T_trans'   : round(T_trans_total,6),
        'E_total'   : round(E_total,6),
        'total_cost': round(total_cost,6),
        'K_reward'  : round(K_tot,6),
        'acc'       : round(acc,4),
    }

# ── Run Case 1 ────────────────────────────────────────────────
print("Case 1 : UAVs=7,  sensors =", case1_sensors)
res1=[]
for ns in case1_sensors:
    t0=time.perf_counter(); r=run_one(ns,case1_uavs)
    r['exec_time']=round(time.perf_counter()-t0,3); res1.append(r)
    print(f"  S={ns:5d} K={r['K']:3d} U=7  | matched={r['matched']}/{r['K']} | "
          f"E_tx={r['E_tx']:.4f} | E_rx={r['E_rx']:.4f} | "
          f"E_tot={r['E_total']:.4f} | K_rew={r['K_reward']:.4f}")
df1=pd.DataFrame(res1)

# ── Run Case 2 ────────────────────────────────────────────────
print("\nCase 2 : UAVs=15, sensors =", case2_sensors)
res2=[]
for ns in case2_sensors:
    t0=time.perf_counter(); r=run_one(ns,case2_uavs)
    r['exec_time']=round(time.perf_counter()-t0,3); res2.append(r)
    print(f"  S={ns:5d} K={r['K']:3d} U=15 | matched={r['matched']}/{r['K']} | "
          f"E_tx={r['E_tx']:.4f} | E_rx={r['E_rx']:.4f} | "
          f"E_tot={r['E_total']:.4f} | K_rew={r['K_reward']:.4f}")
df2=pd.DataFrame(res2)

total_time=time.perf_counter()-overall_start
print(f"\nTotal time: {total_time:.2f}s")
print("\n── Case 1 Summary (UAVs=7) ──")
print(df1[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))
print("\n── Case 2 Summary (UAVs=15) ──")
print(df2[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))

## Plot 1 — K-Means++ Clustering with Voronoi Boundaries

In [ ]:
COLORS_CH  = plt.cm.tab20(np.linspace(0,1,K))
COLORS_UAV = plt.cm.Set1(np.linspace(0,0.85,num_uavs))

fig, ax = plt.subplots(figsize=(9,9))
try:
    vor=Voronoi(kmeans.cluster_centers_)
    voronoi_plot_2d(vor,ax=ax,show_vertices=False,line_colors='gray',
                    line_width=0.8,line_alpha=0.5,point_size=0)
except: pass
sub=df.sample(min(500,len(df)),random_state=1)
ax.scatter(sub["x"],sub["y"],c=sub["cluster_id"],cmap="tab20",s=8,alpha=0.35,label="Sensors")
for h,ch_id in enumerate(CH_list):
    ax.scatter(df.loc[ch_id,"x"],df.loc[ch_id,"y"],color=COLORS_CH[h],s=100,
               marker='D',edgecolors='k',lw=0.8,zorder=4)
ax.plot([0,AREA_SIZE,AREA_SIZE,0,0],[0,0,AREA_SIZE,AREA_SIZE,0],'k-',lw=2)
ax.set_title(f'K-Means++ Clustering with Voronoi Boundaries\nSensors={NUM_SENSORS}, CHs={K}',
             fontweight='bold',fontsize=12)
ax.set_xlabel('X (m)',fontsize=11); ax.set_ylabel('Y (m)',fontsize=11)
ax.legend(fontsize=9); ax.grid(True,alpha=0.2)
plt.tight_layout(); plt.savefig("plot1_kmeans_voronoi.png",dpi=150,bbox_inches='tight')
plt.show(); print("Saved: plot1_kmeans_voronoi.png")

## Plot 2 — CH → UAV Stable Matching

In [ ]:
fig, ax = plt.subplots(figsize=(9,9))
sub=df.sample(min(400,len(df)),random_state=1)
ax.scatter(sub["x"],sub["y"],c=sub["cluster_id"],cmap="tab20",s=6,alpha=0.3,label="Sensors")
for h,ch_id in enumerate(CH_list):
    ax.scatter(df.loc[ch_id,"x"],df.loc[ch_id,"y"],color=COLORS_CH[h],
               s=80,marker='D',edgecolors='k',lw=0.7,zorder=4)
for u in range(num_uavs):
    ux,uy=UAV_positions[u]
    ax.scatter(ux,uy,color=COLORS_UAV[u],s=180,marker='^',edgecolors='k',lw=1.2,zorder=5)
    ax.annotate(f'U{u+1}',(ux,uy),xytext=(3,3),textcoords='offset points',
                fontsize=7,color=COLORS_UAV[u],fontweight='bold')
    for h in M_u[u]:
        cx=df.loc[CH_list[h],"x"]; cy=df.loc[CH_list[h],"y"]
        ax.plot([ux,cx],[uy,cy],color=COLORS_UAV[u],lw=1.8,ls='--',alpha=0.7,zorder=3)
ax.scatter(*UAV_BASE,color='red',s=300,marker='*',edgecolors='k',lw=1.5,zorder=6,label='Base')
ax.plot([0,AREA_SIZE,AREA_SIZE,0,0],[0,0,AREA_SIZE,AREA_SIZE,0],'k-',lw=2)
ax.set_title(f'CH → UAV Stable Matching (Joint Deployment)\nMatched: {n_matched}/{K} CHs',
             fontweight='bold',fontsize=12)
ax.set_xlabel('X (m)',fontsize=11); ax.set_ylabel('Y (m)',fontsize=11)
ax.legend(fontsize=9); ax.grid(True,alpha=0.2)
plt.tight_layout(); plt.savefig("plot2_ch_uav_matching.png",dpi=150,bbox_inches='tight')
plt.show(); print("Saved: plot2_ch_uav_matching.png")

## Plot 3 — Sensor → CH Stable Matching

In [ ]:
fig, ax = plt.subplots(figsize=(9,9))
ax.scatter(df["x"],df["y"],c=df["cluster_id"],cmap="tab20",s=8,alpha=0.4,label="Sensors")
for h,ch_id in enumerate(CH_list):
    hx,hy=df.loc[ch_id,"x"],df.loc[ch_id,"y"]
    ax.scatter(hx,hy,color=COLORS_CH[h],s=120,marker='*',edgecolors='k',lw=0.8,zorder=4)
    for s_id in stable_dict[ch_id]:
        sx,sy=df.loc[s_id,"x"],df.loc[s_id,"y"]
        ax.plot([sx,hx],[sy,hy],lw=2.5,alpha=0.75,color=COLORS_CH[h])
ax.plot([0,AREA_SIZE,AREA_SIZE,0,0],[0,0,AREA_SIZE,AREA_SIZE,0],'k-',lw=2)
ax.set_title('Sensor → CH Stable Matching (Eq.2)',fontweight='bold',fontsize=12)
ax.set_xlabel('X (m)',fontsize=11); ax.set_ylabel('Y (m)',fontsize=11)
ax.legend(fontsize=9); ax.grid(True,alpha=0.2)
plt.tight_layout(); plt.savefig("plot3_sensor_ch_matching.png",dpi=150,bbox_inches='tight')
plt.show(); print("Saved: plot3_sensor_ch_matching.png")

## Plot 4 — UAV Load (CHs per UAV)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
uav_load=[len(M_u[u]) for u in range(num_uavs)]
bars=ax.bar([f'UAV{u+1}' for u in range(num_uavs)],uav_load,
            color=[COLORS_UAV[u] for u in range(num_uavs)],
            edgecolor='k',lw=0.7,alpha=0.85)
for bar,v in zip(bars,uav_load):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.05, str(v),
            ha='center',va='bottom',fontsize=9,fontweight='bold')
ax.set_title('UAV Load: Number of CHs Assigned per UAV (Algorithm 2)',fontweight='bold',fontsize=12)
ax.set_ylabel('Number of CHs',fontsize=11); ax.tick_params(axis='x',labelsize=9,rotation=45)
ax.grid(True,alpha=0.2,axis='y')
plt.tight_layout(); plt.savefig("plot4_uav_load.png",dpi=150,bbox_inches='tight')
plt.show(); print("Saved: plot4_uav_load.png")

## Plot 5 — Transmission Energy E_tx  (independent of UAV count)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['E_tx'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='steelblue',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Transmission Energy E_tx  (independent of UAV count)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors', fontsize=11)
ax.set_ylabel('E_tx (J)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'UAV count has no effect on this metric',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
ax.text(0.98,0.06,"Sensor→CH energy (E_cir + path loss)",transform=ax.transAxes,fontsize=8,ha="right",va="bottom",color="gray")
plt.tight_layout()
plt.savefig('plot5_Etx.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot5_Etx.png')

## Plot 6 — Reception Energy E_rx  (independent of UAV count)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['E_rx'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='darkorange',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Reception Energy E_rx  (independent of UAV count)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors', fontsize=11)
ax.set_ylabel('E_rx (J)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'UAV count has no effect on this metric',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
ax.text(0.98,0.06,"CH receive energy (k_n x E_cir)",transform=ax.transAxes,fontsize=8,ha="right",va="bottom",color="gray")
plt.tight_layout()
plt.savefig('plot6_Erx.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot6_Erx.png')

## Plot 7 — Total System Energy E_total — Case 1  (UAVs=7)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['n_sensors'].tolist()
y_vals = df1['E_total'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='crimson',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Total System Energy E_total — Case 1  (UAVs=7)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 1: UAVs=7, S=200->1000]', fontsize=11)
ax.set_ylabel('E_total (J)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98,0.02,"E_tx+E_rx+E_CH->UAV+E_tve",transform=ax.transAxes,fontsize=8,ha="right",va="bottom",color="gray")
plt.tight_layout()
plt.savefig('plot7_Etotal_case1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot7_Etotal_case1.png')

## Plot 8 — Total System Energy E_total — Case 2  (UAVs=15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['E_total'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='firebrick',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Total System Energy E_total — Case 2  (UAVs=15)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 2: UAVs=15, S=200->2000]', fontsize=11)
ax.set_ylabel('E_total (J)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98,0.02,"E_tx+E_rx+E_CH->UAV+E_tve",transform=ax.transAxes,fontsize=8,ha="right",va="bottom",color="gray")
plt.tight_layout()
plt.savefig('plot8_Etotal_case2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot8_Etotal_case2.png')

## Plot 9 — Total Transmission Time T_trans — Case 1 (UAVs=7)
X-axis: Number of Cluster Heads K (dynamic with sensors)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['K'].tolist()
y_vals = df1['T_trans'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='purple',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Total Transmission Time T_trans — Case 1 (UAVs=7)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Cluster Heads K (dynamic)', fontsize=11)
ax.set_ylabel('T_trans (s)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(k) for k in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'More CHs -> more total Q_h -> higher T_trans',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
sensors = df1['n_sensors'].tolist()
for i, (k, s) in enumerate(zip(x_vals, sensors)):
    ax.text(i, -ax.get_ylim()[1]*0.06, f'N={s}',
            ha='center', fontsize=7, color='dimgray')
ax.text(0.02, -0.08, 'Sensors:', transform=ax.transAxes,
        fontsize=7, color='dimgray')
plt.tight_layout()
plt.savefig('plot9_Ttrans_case1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot9_Ttrans_case1.png')

## Plot 10 — Total Transmission Time T_trans — Case 2 (UAVs=15)
X-axis: Number of Cluster Heads K (dynamic with sensors)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['K'].tolist()
y_vals = df2['T_trans'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='mediumvioletred',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Total Transmission Time — Case 2 (UAVs=15)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Cluster Heads K (dynamic)', fontsize=11)
ax.set_ylabel('T_trans (s)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(k) for k in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'More CHs -> more total Q_h -> higher T_trans',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
sensors = df2['n_sensors'].tolist()
for i, (k, s) in enumerate(zip(x_vals, sensors)):
    ax.text(i, -ax.get_ylim()[1]*0.06, f'N={s}',
            ha='center', fontsize=7, color='dimgray')
ax.text(0.02, -0.08, 'Sensors:', transform=ax.transAxes,
        fontsize=7, color='dimgray')
plt.tight_layout()
plt.savefig('plot10_Ttrans_case2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot10_Ttrans_case2.png')

## Plot 11 — Total Cost — Case 1  (UAVs=7)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['n_sensors'].tolist()
y_vals = df1['total_cost'].tolist()
ax.plot(x_vals, y_vals, marker='o', lw=3, ms=10,
        color='saddlebrown', label='Total Cost')
ax.fill_between(x_vals, y_vals, alpha=0.12, color='saddlebrown')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=9, fontweight='bold')
ax.set_title('Total Cost — Case 1  (UAVs=7)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 1: UAVs=7, N=200->1000]', fontsize=11)
ax.set_ylabel('Total Cost', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98,0.02,"lambda_e x E_total + lambda_t x T_trans",transform=ax.transAxes,fontsize=8,ha="right",va="bottom",color="gray")
plt.tight_layout()
plt.savefig('plot11_cost_case1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot11_cost_case1.png')

## Plot 12 — Total Cost — Case 2  (UAVs=15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['total_cost'].tolist()
ax.plot(x_vals, y_vals, marker='o', lw=3, ms=10,
        color='chocolate', label='Total Cost')
ax.fill_between(x_vals, y_vals, alpha=0.12, color='chocolate')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=9, fontweight='bold')
ax.set_title('Total Cost — Case 2  (UAVs=15)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 2: UAVs=15, N=200->2000]', fontsize=11)
ax.set_ylabel('Total Cost', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98,0.02,"lambda_e x E_total + lambda_t x T_trans",transform=ax.transAxes,fontsize=8,ha="right",va="bottom",color="gray")
plt.tight_layout()
plt.savefig('plot12_cost_case2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot12_cost_case2.png')

## Plot 13 — Total Reward K (Eq.25) — Case 1  (UAVs=7)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['n_sensors'].tolist()
y_vals = df1['K_reward'].tolist()
ax.plot(x_vals, y_vals, marker='o', lw=3, ms=10,
        color='darkgreen', label='K = Sum G_u')
ax.fill_between(x_vals, y_vals, alpha=0.12, color='darkgreen')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=9, fontweight='bold')
ax.set_title('Total Revenue K (Eq.25) — Case 1  (UAVs=7)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 1: UAVs=7, N=200->1000]', fontsize=11)
ax.set_ylabel('K = Sum G_u', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98,0.02,"G_u = R_u - pi_u(E_tare+E_cmp+E_com)",transform=ax.transAxes,fontsize=8,ha="right",va="bottom",color="gray")
plt.tight_layout()
plt.savefig('plot13_reward_case1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot13_reward_case1.png')

## Plot 14 — Total Reward K (Eq.25) — Case 2  (UAVs=15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['K_reward'].tolist()
ax.plot(x_vals, y_vals, marker='o', lw=3, ms=10,
        color='seagreen', label='K = Sum G_u')
ax.fill_between(x_vals, y_vals, alpha=0.12, color='seagreen')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=9, fontweight='bold')
ax.set_title('Total Revenue K (Eq.25) — Case 2  (UAVs=15)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 2: UAVs=15, S=200->2000]', fontsize=11)
ax.set_ylabel('K = Sum G_u', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98,0.02,"G_u = R_u - pi_u(E_tare+E_cmp+E_com)",transform=ax.transAxes,fontsize=8,ha="right",va="bottom",color="gray")
plt.tight_layout()
plt.savefig('plot14_reward_case2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot14_reward_case2.png')

## Plot 15 — FL Accuracy over Rounds (per-round training accuracy)

In [ ]:
if 'fl_acc_rounds' in dir() and len(fl_acc_rounds) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    rounds = list(range(1, len(fl_acc_rounds) + 1))
    ax.plot(rounds, fl_acc_rounds, marker='o', lw=3, ms=10,
            color='royalblue', label='FL Accuracy per Round')
    ax.fill_between(rounds, fl_acc_rounds, alpha=0.12, color='royalblue')
    for r, a in zip(rounds, fl_acc_rounds):
        ax.annotate(f'{a:.4f}', (r, a), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=10, fontweight='bold')
    ax.set_title('FL Accuracy over Training Rounds — Proposed (Stable Matching)',
                 fontweight='bold', fontsize=13)
    ax.set_xlabel('FL Round', fontsize=11)
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_xticks(rounds)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', labelsize=10)
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
    ax.text(0.98, 0.02, f'FedAvg | {NUM_ROUNDS} rounds | {LOCAL_EPOCHS} local epochs',
            transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
    plt.tight_layout()
    plt.savefig('plot15a_fl_acc_rounds.png', dpi=150, bbox_inches='tight')
    plt.show(); print('Saved: plot15a_fl_acc_rounds.png')
else:
    print("fl_acc_rounds not found — run Cell 8 (FL training) first.")

fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['n_sensors'].tolist()
y_vals = df1['acc'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='teal',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('FL Accuracy — Case 1 (UAVs=7)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 1: UAVs=7, S=200->1000]', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')
ax.legend(['Proposed (Stable Matching)'], fontsize=9)
plt.tight_layout()
plt.savefig('plot15b_acc_case1.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: plot15b_acc_case1.png')

fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['acc'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='cadetblue',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('FL Accuracy — Case 2 (UAVs=15)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 2: UAVs=15, S=200->2000]', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')
ax.legend(['Proposed (Stable Matching)'], fontsize=9)
plt.tight_layout()
plt.savefig('plot15c_acc_case2.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: plot15c_acc_case2.png')

## Final Summary

In [ ]:
print("=" * 68)
print("  FINAL SIMULATION SUMMARY")
print("=" * 68)
print(f"  Farm: {AREA_SIZE}x{AREA_SIZE}m | FL rounds: {NUM_ROUNDS} | Epochs: {LOCAL_EPOCHS}")
print(f"  UAV Deployment: Joint K-Means on CH positions (NOT random)")
print()
print("  NOTE: E_tx and E_rx are UAV-independent (Plot 5 & 6 use all sensor values)")
print()
print("  Case 1 — UAVs=7,  Sensors=200,400,600,800,1000  (K dynamic)")
print(df1[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))
print()
print("  Case 2 — UAVs=15, Sensors=200->2000 step 200     (K dynamic)")
print(df2[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))
print("=" * 68)